# 🧪 Classification pipeline with RDF

### **About This Exercise**

Today you will not have any tasks to solve instead you should solve this challenge completely to practice what we have learned so far.

You’ll work with `data.csv`, which contain clinical and pathological data for cancer diagnoses. Each case includes measurements of three cell nuclei (radius, texture, perimeter), along with patient age, diagnosis date, treatment start date, and cancer type.

### **Context**

The task is to classify tissue samples as cancerous or healthy using features extracted by a pathologist. You'll build a machine learning pipeline that preprocesses this data and applies classification models to make accurate predictions. You should finally build a ML for the production phase.


### **Goal**

Your objectives:

* Preprocess the data with various cleaning and transformation methods.
* Build and tune a Random Forest (RDF) classifier.
* Experiment with different hyperparameter settings
* try to reduce n_estimators to 10 , what will happen?
* you must understand all RDF options so please spend some time reading the function signature

To ensure reproducibility, document your preprocessing and modeling options early in the notebook. Recommended practices include:

* **Dictionary in Notebook**: Simple and effective for small projects.
* **JSON Config File**: Useful for managing and reusing configurations.
* **MLflow Tracking**: Best for logging experiments, metrics, and comparing results visually.

These practices support consistent and trackable machine learning workflows.

#### Dictionarys you can use to save the options for the preprocessing and modeling. 

In [1]:
# Configuration dictionary to track preprocessing and modeling choices
config = {
    "missing_value_strategy": "replace_by_mean_cancer_type",  # Options: replace_by_med_cancer_type, replace_by_mean_cancer_type
    "scaling_method": "StandardScaler",                       # Options: StandardScaler, RobustScaler, normalizer, MinMaxScaler
    "RDF": {"n_estimators": 1000, # options: 5, 10, 100
            "max_depth": 6,       # options: None, 2, 6
            "min_samples_split":2, # options: 2, 4, 10
            "min_samples_leaf":2, # options: 2, 4, 10
            "max_features":0.7,
            "class_weight":'balanced'}                         
}

#### Improt needed packages and set the random seed

In [2]:
# Data manipulation and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Model and preprocessing tools will be added as needed later


In [3]:
# Set global random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

#### Read and check the data

In [7]:
data = pd.read_csv('../data/ex6_data.csv',header='infer')
data.head()

,radius_0,texture_0,perimeter_0,radius_1,texture_1,perimeter_1,radius_2,texture_2,perimeter_2,age,treatment_date,diagnose_date,cancer_type
0,19.858394,27.204437,136.324256,22.683290,32.802578,119.523841,21.477052,27.3070874472,82.366936,44,2006-06-03,2005-10-23,0
1,14.182069,15.754730,80.916983,14.043753,30.094704,94.911073,15.012329,17.8551305385,103.078286,59,2004-02-22,2007-08-20,1
2,25.380268,21.291553,152.281062,23.852166,46.237931,NaN,28.563252,21.0971528265,143.367792,37,2006-01-06,2004-08-07,0
3,11.835961,17.820702,72.178523,11.260258,44.805167,NaN,12.082749,16.4992370844,65.920413,51,2003-04-14,2005-06-16,1
4,14.875600,17.534187,98.545830,14.380683,26.190447,89.712492,12.930685,19.8566873539,108.380754,21,2004-06-21,2002-11-27,1


In [26]:
# Format texture_2 to float64
print(data.dtypes)
data["texture_2"] = data["texture_2"].replace(to_replace="xx", value=np.nan)
data["texture_2"] = data["texture_2"].astype("float64")

radius_0          float64
texture_0         float64
perimeter_0       float64
radius_1          float64
texture_1         float64
perimeter_1       float64
radius_2          float64
texture_2             str
perimeter_2       float64
age                 int64
treatment_date        str
diagnose_date         str
cancer_type         int64
dtype: object


In [29]:
# Cast treatment_date and diagnose_date to proper data types
print(data.dtypes)
data["treatment_date"] = pd.to_datetime(data["treatment_date"])
data["diagnose_date"] = pd.to_datetime(data["diagnose_date"])

radius_0                 float64
texture_0                float64
perimeter_0              float64
radius_1                 float64
texture_1                float64
perimeter_1              float64
radius_2                 float64
texture_2                float64
perimeter_2              float64
age                        int64
treatment_date    datetime64[us]
diagnose_date     datetime64[us]
cancer_type                int64
dtype: object


In [41]:
# Impute missing values
for col in data.columns[data.isna().sum() > 0]:
    data[col] = data[col].fillna(data[col].median())

data.isna().sum()

radius_0          0
texture_0         0
perimeter_0       0
radius_1          0
texture_1         0
perimeter_1       0
radius_2          0
texture_2         0
perimeter_2       0
age               0
treatment_date    0
diagnose_date     0
cancer_type       0
dtype: int64

In [52]:
# Extract features from Datetime columns
data["treatment_year"] = data["treatment_date"].dt.year
data["treatment_month"] = data["treatment_date"].dt.month
data["treatment_day"] = data["treatment_date"].dt.day

data["diagnose_year"] = data["diagnose_date"].dt.year
data["diagnose_month"] = data["diagnose_date"].dt.month
data["diagnose_day"] = data["diagnose_date"].dt.day

data.head()

,radius_0,texture_0,perimeter_0,radius_1,texture_1,perimeter_1,radius_2,texture_2,perimeter_2,age,treatment_date,diagnose_date,cancer_type,treatment_year,treatment_month,treatment_day,diagnose_year,diagnose_month,diagnose_day
0,19.858394,27.204437,136.324256,22.683290,32.802578,119.523841,21.477052,27.307087,82.366936,44,2006-06-03,2005-10-23,0,2006,6,3,2005,10,23
1,14.182069,15.754730,80.916983,14.043753,30.094704,94.911073,15.012329,17.855131,103.078286,59,2004-02-22,2007-08-20,1,2004,2,22,2007,8,20
2,25.380268,21.291553,152.281062,23.852166,46.237931,90.054613,28.563252,21.097153,143.367792,37,2006-01-06,2004-08-07,0,2006,1,6,2004,8,7
3,11.835961,17.820702,72.178523,11.260258,44.805167,90.054613,12.082749,16.499237,65.920413,51,2003-04-14,2005-06-16,1,2003,4,14,2005,6,16
4,14.875600,17.534187,98.545830,14.380683,26.190447,89.712492,12.930685,19.856687,108.380754,21,2004-06-21,2002-11-27,1,2004,6,21,2002,11,27


In [58]:
# Drop original datetime columns
data.drop(["treatment_date", "diagnose_date"], axis=1, inplace=True)
data.head()

,radius_0,texture_0,perimeter_0,radius_1,texture_1,perimeter_1,radius_2,texture_2,perimeter_2,age,cancer_type,treatment_year,treatment_month,treatment_day,diagnose_year,diagnose_month,diagnose_day
0,19.858394,27.204437,136.324256,22.683290,32.802578,119.523841,21.477052,27.307087,82.366936,44,0,2006,6,3,2005,10,23
1,14.182069,15.754730,80.916983,14.043753,30.094704,94.911073,15.012329,17.855131,103.078286,59,1,2004,2,22,2007,8,20
2,25.380268,21.291553,152.281062,23.852166,46.237931,90.054613,28.563252,21.097153,143.367792,37,0,2006,1,6,2004,8,7
3,11.835961,17.820702,72.178523,11.260258,44.805167,90.054613,12.082749,16.499237,65.920413,51,1,2003,4,14,2005,6,16
4,14.875600,17.534187,98.545830,14.380683,26.190447,89.712492,12.930685,19.856687,108.380754,21,1,2004,6,21,2002,11,27


In [61]:
# Compute the train/test split
from sklearn.model_selection import train_test_split
target = "cancer_type"

y = data[target]
X = data.drop(labels=(target), axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

In [77]:
# Standardize the features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
print(scaler.fit(X_train))

X_train_scaled_np = scaler.transform(X_train)
X_test_scaled_np = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(data=X_train_scaled_np, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(data=X_test_scaled_np, columns=X_test.columns, index=X_test.index)

StandardScaler()


[[-0.18739556 -0.16404375 -0.78141405 ...  0.3708799  -0.05817818
   0.30749009]
 [-0.17841176 -0.52580798 -0.49771848 ...  0.3708799  -0.37715511
   0.9637441 ]
 [-0.07006631 -0.84030374  1.33129315 ...  1.07290258 -1.65306282
  -0.74251633]
 ...
 [-0.10422713  1.11879406  0.89528475 ...  1.07290258  1.53670646
  -0.74251633]
 [-0.23519263 -0.77684133 -1.04586568 ... -0.33114277  0.8987526
  -0.74251633]
 [-0.14036934 -0.25866675 -0.434275   ...  0.3708799  -1.01510896
   0.83249329]]


In [97]:
# Create the random forest
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(n_estimators=10, random_state=RANDOM_STATE, max_depth=15)
clf.fit(X_train_scaled, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",10
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

In [99]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

pred_train = clf.predict(X_train_scaled)
pred_test = clf.predict(X_test_scaled)


print(f"Accuracy Score Train: {accuracy_score(y_train, pred_train)}")
print(f"Accuracy Score Test : {accuracy_score(y_test, pred_test)}")

print(f"F1 Score Train: {f1_score(y_train, pred_train)}")
print(f"F1 Score Test : {f1_score(y_test, pred_test)}")

print(f"ROC-AUC Curve Train: {roc_auc_score(y_train, pred_train)}")
print(f"ROC-AUC Curve Test : {roc_auc_score(y_test, pred_test)}")

Accuracy Score Train: 0.9874213836477987
Accuracy Score Test : 0.8625
F1 Score Train: 0.9900497512437811
F1 Score Test : 0.8910891089108911
ROC-AUC Curve Train: 0.9882639125981564
ROC-AUC Curve Test : 0.84375
